# DATA INGESTION

In [1]:
from pathlib import Path
from typing import List
import re

from langchain_core.documents import Document

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from bs4 import BeautifulSoup

c:\Users\boroh\ELTE\Thesis\elte_chat\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


## File Loaders

In [2]:
def clean_text(text: str) -> str:
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def load_pdf(file_path: str | Path) -> List[Document]:
    file_path = Path(file_path)
    loader = PyPDFLoader(str(file_path))
    docs = loader.load()

    normalized_docs = []
    for i, doc in enumerate(docs):
        normalized_docs.append(
            Document(
                page_content=clean_text(doc.page_content),
                metadata={
                    "source": str(file_path),
                    "file_name": file_path.name,
                    "file_type": "pdf",
                    "page": doc.metadata.get("page", i),
                    **doc.metadata
                }
            )
        )

    return normalized_docs


def load_pdf_pymupdf(file_path: str | Path) -> List[Document]:
    file_path = Path(file_path)
    pdf = pymupdf.open(str(file_path))

    docs = []
    for page_num, page in enumerate(pdf):
        text = page.get_text("text")
        docs.append(
            Document(
                page_content=clean_text(text),
                metadata={
                    "source": str(file_path),
                    "file_name": file_path.name,
                    "file_type": "pdf",
                    "page": page_num
                }
            )
        )

    pdf.close()
    return docs


def load_txt(file_path: str | Path, encoding: str = "utf-8") -> List[Document]:
    file_path = Path(file_path)
    loader = TextLoader(str(file_path), encoding=encoding)
    docs = loader.load()

    normalized_docs = []
    for doc in docs:
        normalized_docs.append(
            Document(
                page_content=clean_text(doc.page_content),
                metadata={
                    "source": str(file_path),
                    "file_name": file_path.name,
                    "file_type": "txt",
                    **doc.metadata
                }
            )
        )

    return normalized_docs


def load_html(file_path: str | Path) -> List[Document]:
    file_path = Path(file_path)

    with open(file_path, "r", encoding="utf-8") as f:
        html = f.read()

    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    text = "\n".join(line.strip() for line in text.splitlines() if line.strip())

    return [
        Document(
            page_content=clean_text(text),
            metadata={
                "source": str(file_path),
                "file_name": file_path.name,
                "file_type": "html",
                "title": soup.title.string.strip() if soup.title and soup.title.string else None
            }
        )
    ]

In [3]:
def load_file(file_path: str | Path) -> List[Document]:
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()

    if suffix == ".pdf":
        return load_pdf(file_path)
    elif suffix in {".txt", ".md"}:
        return load_txt(file_path)
    elif suffix in {".html", ".htm"}:
        return load_html(file_path)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")
    
def load_directory(directory: str | Path) -> List[Document]:
    directory = Path(directory)
    all_docs = []

    for file_path in directory.rglob("*"):
        if file_path.is_file():
            try:
                docs = load_file(file_path)
                all_docs.extend(docs)
                print(f"Loaded: {file_path}")
            except Exception as e:
                print(f"Skipped {file_path}: {e}")

    return all_docs

## Chunking

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    length_function=len,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)


def save_chunks(chunks, path="../data/processed/chunks.json"):
    data = []

    for doc in chunks:
        data.append({
            "content": doc.page_content,
            "metadata": doc.metadata
        })

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

## Storing the processed data in json

In [7]:
MIN_CHUNK_SIZE = 100

docs = load_directory("../data/raw")
chunked_docs = text_splitter.split_documents(docs)
chunked_docs = [doc for doc in chunked_docs if len(doc.page_content) >= MIN_CHUNK_SIZE]
for i, doc in enumerate(chunked_docs):
    doc.metadata["chunk_id"] = i

save_chunks(chunked_docs)
print(f"Saved {len(chunked_docs)} chunks")

Loaded: ..\data\raw\ELTE Faculty of Informatics.html
Loaded: ..\data\raw\The prerequisites.pdf
Loaded: ..\data\raw\About us\Dean & Administration _ ELTE Faculty of Informatics.html
Loaded: ..\data\raw\About us\Department of Academic and International Relations _ ELTE Faculty of Informatics.pdf
Loaded: ..\data\raw\About us\History of the faculty _ ELTE Faculty of Informatics.html
Loaded: ..\data\raw\About us\Institute of Cartography and Geoinformatics _ ELTE Faculty of Informatics.html
Loaded: ..\data\raw\About us\Institute of Computer Science _ ELTE Faculty of Informatics.html


invalid pdf header: b'PK\x03\x04\x14'
EOF marker not found


Loaded: ..\data\raw\About us\Institute of Industry-Academia Innovation _ ELTE Faculty of Informatics.html
Loaded: ..\data\raw\About us\Student Support Centre _ ELTE Faculty of Informatics.pdf
Loaded: ..\data\raw\About us\Why choose us_ _ ELTE Faculty of Informatics.html
Skipped ..\data\raw\ELTE Faculty of Informatics_files\accessibility-loader(1).js.download: Unsupported file type: .download
Skipped ..\data\raw\ELTE Faculty of Informatics_files\accessibility-loader.js.download: Unsupported file type: .download
Skipped ..\data\raw\ELTE Faculty of Informatics_files\algoliasearch-lite.umd.js.download: Unsupported file type: .download
Skipped ..\data\raw\ELTE Faculty of Informatics_files\all-in-one-accessibility-js-widget-minify.js.download: Unsupported file type: .download
Skipped ..\data\raw\ELTE Faculty of Informatics_files\css_RyNX8ZPieUc8XSexUVr62heq_oKZCIdyC6xHPwLiHkI.css: Unsupported file type: .css
Skipped ..\data\raw\ELTE Faculty of Informatics_files\css_WG8WG520JWHi27qdYwgMDtuLgI

## Linked PDF Downloader

In [6]:
import requests
from urllib.parse import urljoin, urlparse

def download_linked_pdfs(
    html_dir: str | Path,
    out_dir: str | Path,
    base_url: str = ""
) -> list[Path]:
    html_dir = Path(html_dir)
    out_dir  = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    downloaded = []

    for html_file in html_dir.rglob("*.html"):
        soup = BeautifulSoup(html_file.read_text(encoding="utf-8"), "html.parser")
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if not href.lower().endswith(".pdf"):
                continue

            url = urljoin(base_url, href) if base_url else href
            if not url.startswith("http"):
                print(f"  Skipped (relative, no base_url): {href}")
                continue

            filename = Path(urlparse(url).path).name
            dest = out_dir / filename

            if dest.exists():
                print(f"  Already exists, skipping: {filename}")
                continue

            try:
                r = requests.get(url, timeout=30)
                r.raise_for_status()
                dest.write_bytes(r.content)
                print(f"  Downloaded: {filename}")
                downloaded.append(dest)
            except Exception as e:
                print(f"  Failed {url}: {e}")

    return downloaded


# Run it — safe to re-run, skips already-downloaded files
new_files = download_linked_pdfs(
    html_dir="../data/raw",
    out_dir="../data/raw/linked_pdfs",
)
print(f"Downloaded {len(new_files)} new PDF(s)")
if new_files:
    print("Re-run ingestion cells above to include them in chunks.json")

  Downloaded: Who%20to%20contact.pdf
  Already exists, skipping: Who%20to%20contact.pdf
  Already exists, skipping: Who%20to%20contact.pdf
  Already exists, skipping: Who%20to%20contact.pdf
  Already exists, skipping: Who%20to%20contact.pdf
  Already exists, skipping: Who%20to%20contact.pdf
  Already exists, skipping: Who%20to%20contact.pdf
  Downloaded: acc%20reporting%20form%202023%20EN.docx.pdf
  Downloaded: Job%20seeking%20permit%20guide.pdf
  Downloaded: OIF%20Information_ENG.pdf
  Downloaded: Notification%20of%20accommodation.Fill%20in%20guide.pdf
  Downloaded: Enter%20Hungary%20Guide%202023.03..pdf
Downloaded 6 new PDF(s)
Re-run ingestion cells above to include them in chunks.json
